# Experiment 2: Template sensitivity

This notebook reads **passwords** and **templates** from CSV files. The templates are not hard-coded in Python string literals, so multiline prompts can be edited in the CSV without causing syntax errors.
Unfortunately, the direct-judgement template comparison result from my ECE 202C presentation is shockingly hard to reproduce. I apologize for the inconvenience.
Inputs:
- `experiment2_template_sensitivity_passwords.csv`
- `experiment2_templates.csv`

Main outputs go to `results_exp2/`.


In [ ]:
# Install deps if needed.
import sys, subprocess, os, math, time, gc, json, re
from pathlib import Path

for pkg in ["transformers", "accelerate", "zxcvbn", "pandas", "matplotlib", "tqdm", "scikit-learn"]:
    try:
        __import__(pkg.replace("-", "_"))
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from zxcvbn import zxcvbn
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from sklearn.metrics import roc_auc_score
except Exception:
    roc_auc_score = None

RESULTS_DIR = Path("results_exp2")
FIG_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)


In [ ]:
# Input files. In Colab, upload these two files or keep them beside the notebook.
PASSWORD_DATA_PATH = "experiment2_template_sensitivity_passwords.csv"
TEMPLATE_DATA_PATH = "experiment2_templates.csv"

# Model. Template sweep uses Qwen 1.5B. Run final selected template on Mistral separately if desired.
MODEL_ALIAS = "qwen25_1p5b"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
TRUST_REMOTE_CODE = True

# Runtime controls.
RUN_CONTINUATION = True
RUN_STRUCTURAL = True
RUN_FIRST_N_PASSWORDS = None   # set to e.g. 20 for a smoke test
RUN_FIRST_N_TEMPLATES_PER_TASK = None  # set to e.g. 3 for a smoke test

# DP controls. Scores were stable at small beams in earlier runs; increase for final if desired.
BEAM_PER_INDEX = 32
MAX_VALID_TOKENS_PER_STATE = 32
BATCH_STATES = 32

# Structural yes/no completion variants.
YES_STRINGS = [" yes", " Yes", "yes", "Yes"]
NO_STRINGS  = [" no", " No", "no", "No"]

print("password file:", PASSWORD_DATA_PATH)
print("template file:", TEMPLATE_DATA_PATH)
print("model:", MODEL_ID)


In [ ]:
# Load inputs.
df_pw = pd.read_csv(PASSWORD_DATA_PATH)
df_templates = pd.read_csv(TEMPLATE_DATA_PATH)

required_pw_cols = {"password_id", "family", "condition", "group_id", "password", "chunks", "known_prefix_chunks", "expected_structural_label", "use_for_continuation_dp"}
required_tpl_cols = {"template_id", "task", "group", "text", "enabled"}
missing_pw = required_pw_cols - set(df_pw.columns)
missing_tpl = required_tpl_cols - set(df_templates.columns)
assert not missing_pw, f"Password CSV missing columns: {missing_pw}"
assert not missing_tpl, f"Template CSV missing columns: {missing_tpl}"

# Normalize booleans loaded from CSV.
def as_bool(x):
    if isinstance(x, bool):
        return x
    return str(x).strip().lower() in {"true", "1", "yes", "y"}

df_pw["use_for_continuation_dp"] = df_pw["use_for_continuation_dp"].map(as_bool)
df_templates["enabled"] = df_templates["enabled"].map(as_bool)

df_pw = df_pw.copy()
if RUN_FIRST_N_PASSWORDS is not None:
    df_pw = df_pw.head(RUN_FIRST_N_PASSWORDS).copy()

df_templates_enabled = df_templates[df_templates["enabled"]].copy()
if RUN_FIRST_N_TEMPLATES_PER_TASK is not None:
    df_templates_enabled = (
        df_templates_enabled.groupby("task", group_keys=False)
        .head(RUN_FIRST_N_TEMPLATES_PER_TASK)
        .copy()
    )

continuation_templates = df_templates_enabled[df_templates_enabled["task"] == "continuation"].to_dict("records")
structural_templates = df_templates_enabled[df_templates_enabled["task"] == "structural"].to_dict("records")

print("password rows:", len(df_pw))
print("continuation rows:", int(df_pw["use_for_continuation_dp"].sum()))
print("continuation templates:", len(continuation_templates))
display(pd.DataFrame(continuation_templates)[["template_id", "group", "text"]])
print("structural templates:", len(structural_templates))
display(pd.DataFrame(structural_templates)[["template_id", "group", "text"]])


In [ ]:
# Load model.
print("Loading", MODEL_ID)
t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=TRUST_REMOTE_CODE,
)
model.eval()
print(f"Loaded in {time.perf_counter() - t0:.1f}s")
print("device:", next(model.parameters()).device)


In [ ]:
# Shared helpers.
def safe_format_template(text, *, prefix=None, password=None):
    # Keep it boring: templates should only use {prefix} and/or {password}.
    if prefix is None:
        prefix = ""
    if password is None:
        password = ""
    return str(text).format(prefix=prefix, password=password)

def zxcvbn_log10(password):
    r = zxcvbn(str(password))
    g = float(r["guesses"])
    return math.log10(g) if g > 0 else float("-inf")

def parse_chunks(row):
    chunks = [c for c in str(row["chunks"]).split("|") if c != ""]
    k = int(row["known_prefix_chunks"])
    k = max(0, min(k, len(chunks)))
    prefix = "".join(chunks[:k])
    remainder = "".join(chunks[k:])
    return chunks, prefix, remainder

def logsumexp(vals):
    vals = list(vals)
    if not vals:
        return -float("inf")
    m = max(vals)
    if not math.isfinite(m):
        return m
    return m + math.log(sum(math.exp(v - m) for v in vals))


In [ ]:
# Build token decode index for constrained continuation DP.
# This maps first character -> [(token_id, decoded_piece), ...].
# It avoids scanning the full vocabulary for every DP state.
print("Building token decode index...")
t0 = time.perf_counter()
TOKEN_BY_FIRST_CHAR = {}
ALL_VALID_TOKEN_PIECES = []
for tid in range(len(tokenizer)):
    try:
        piece = tokenizer.decode([tid], clean_up_tokenization_spaces=False)
    except Exception:
        continue
    if piece is None or piece == "":
        continue
    if "\ufffd" in piece:
        continue
    # We do not globally reject spaces/punctuation because the target decides validity.
    first = piece[0]
    TOKEN_BY_FIRST_CHAR.setdefault(first, []).append((tid, piece))
    ALL_VALID_TOKEN_PIECES.append((tid, piece))
print(f"indexed {len(ALL_VALID_TOKEN_PIECES)} token pieces in {time.perf_counter() - t0:.1f}s")


In [ ]:
@torch.no_grad()
def batched_next_logprobs(prompts):
    enc = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model(**enc, use_cache=False)
    attn = enc["attention_mask"]
    last_pos = attn.sum(dim=1) - 1
    logits = out.logits[torch.arange(len(prompts), device=out.logits.device), last_pos, :]
    return torch.log_softmax(logits.float(), dim=-1).cpu()

def valid_next_tokens(remaining):
    if remaining == "":
        return []
    candidates = TOKEN_BY_FIRST_CHAR.get(remaining[0], [])
    out = []
    for tid, piece in candidates:
        if remaining.startswith(piece):
            out.append((tid, piece))
    return out

def constrained_remainder_log10(rendered_prompt, target_remainder,
                                beam_per_index=BEAM_PER_INDEX,
                                max_valid_tokens_per_state=MAX_VALID_TOKENS_PER_STATE,
                                batch_states=BATCH_STATES):
    """Approximate -log10 P(target_remainder | rendered_prompt) by summing valid token paths.

    DP state at character index i stores tokenization paths whose decoded text equals target_remainder[:i].
    Each path is (logprob_natural, generated_text). We keep a beam per character index.
    """
    target_remainder = str(target_remainder)
    n = len(target_remainder)
    if n == 0:
        return dict(llm_remainder_log10=0.0, dp_status="empty", num_states_expanded=0, num_finished_paths=1, truncated=False)

    # dp[i] = list of (logprob, generated_prefix)
    dp = [[] for _ in range(n + 1)]
    dp[0] = [(0.0, "")]
    num_states_expanded = 0
    truncated = False

    for i in range(n):
        if not dp[i]:
            continue
        # Beam at this index by current logprob.
        dp[i].sort(key=lambda x: x[0], reverse=True)
        if len(dp[i]) > beam_per_index:
            dp[i] = dp[i][:beam_per_index]
            truncated = True

        states = dp[i]
        remaining = target_remainder[i:]
        candidates = valid_next_tokens(remaining)
        if not candidates:
            continue

        # Score states in batches.
        for b0 in range(0, len(states), batch_states):
            batch = states[b0:b0+batch_states]
            prompts = [rendered_prompt + gen for _, gen in batch]
            logprobs_batch = batched_next_logprobs(prompts)
            num_states_expanded += len(batch)

            for row_idx, (base_lp, gen) in enumerate(batch):
                cand_scores = []
                lp_row = logprobs_batch[row_idx]
                for tid, piece in candidates:
                    cand_scores.append((float(lp_row[tid].item()), tid, piece))
                cand_scores.sort(key=lambda x: x[0], reverse=True)
                if len(cand_scores) > max_valid_tokens_per_state:
                    cand_scores = cand_scores[:max_valid_tokens_per_state]
                    truncated = True
                for lp_token, tid, piece in cand_scores:
                    j = i + len(piece)
                    dp[j].append((base_lp + lp_token, gen + piece))

        # Trim all future buckets to avoid blowup.
        for j in range(i + 1, n + 1):
            if len(dp[j]) > beam_per_index:
                dp[j].sort(key=lambda x: x[0], reverse=True)
                dp[j] = dp[j][:beam_per_index]
                truncated = True

    finished = dp[n]
    if not finished:
        return dict(llm_remainder_log10=float("inf"), dp_status="no_path", num_states_expanded=num_states_expanded, num_finished_paths=0, truncated=truncated)

    total_logp = logsumexp([lp for lp, _ in finished])
    return dict(
        llm_remainder_log10=-total_logp / math.log(10),
        dp_status="ok",
        num_states_expanded=num_states_expanded,
        num_finished_paths=len(finished),
        truncated=truncated,
    )


In [ ]:
# Run continuation-template sweep.
continuation_rows = []
if RUN_CONTINUATION:
    df_cont = df_pw[df_pw["use_for_continuation_dp"]].copy()
    total = len(df_cont) * len(continuation_templates)
    pbar = tqdm(total=total, desc="Continuation DP")
    t0 = time.perf_counter()

    for _, row in df_cont.iterrows():
        chunks, prefix, remainder = parse_chunks(row)
        if remainder == "":
            continue
        zx = zxcvbn_log10(row["password"])
        for tpl in continuation_templates:
            rendered = safe_format_template(tpl["text"], prefix=prefix, password=row["password"])
            res = constrained_remainder_log10(rendered, remainder)
            continuation_rows.append({
                "model_alias": MODEL_ALIAS,
                "password_id": row["password_id"],
                "family": row["family"],
                "condition": row["condition"],
                "group_id": row["group_id"],
                "password": row["password"],
                "chunks": row["chunks"],
                "known_prefix_chunks": int(row["known_prefix_chunks"]),
                "known_prefix": prefix,
                "target_remainder": remainder,
                "zxcvbn_guesses_log10": zx,
                "template_id": tpl["template_id"],
                "template_group": tpl["group"],
                "template_text": tpl["text"],
                **res,
            })
            pbar.update(1)
    pbar.close()
    print(f"Continuation sweep finished in {time.perf_counter() - t0:.1f}s")

    df_cont_raw = pd.DataFrame(continuation_rows)
    df_cont_raw.to_csv(RESULTS_DIR / "exp2_continuation_raw.csv", index=False)
    display(df_cont_raw.head())
else:
    df_cont_raw = pd.DataFrame()
    print("RUN_CONTINUATION=False")


In [ ]:
# Continuation summaries.
if len(df_cont_raw):
    cond_summary = (
        df_cont_raw.groupby(["template_id", "template_group", "condition"])
        .agg(
            n=("password_id", "count"),
            avg_llm_remainder_log10=("llm_remainder_log10", "mean"),
            median_llm_remainder_log10=("llm_remainder_log10", "median"),
            avg_zxcvbn_log10=("zxcvbn_guesses_log10", "mean"),
            trunc_rate=("truncated", "mean"),
        )
        .reset_index()
    )
    cond_summary.to_csv(RESULTS_DIR / "exp2_continuation_template_condition_summary.csv", index=False)
    display(cond_summary)

    # Paired linked/control comparisons by group_id when both conditions exist.
    pair_rows = []
    for (tpl_id, gid), g in df_cont_raw.groupby(["template_id", "group_id"]):
        linked = g[g["condition"].astype(str).str.contains("linked")]
        control = g[g["condition"].astype(str).str.contains("control")]
        if len(linked) and len(control):
            l = linked.iloc[0]
            c = control.iloc[0]
            pair_rows.append({
                "template_id": tpl_id,
                "template_group": l["template_group"],
                "group_id": gid,
                "family": l["family"],
                "linked_password": l["password"],
                "control_password": c["password"],
                "linked_llm_log10": l["llm_remainder_log10"],
                "control_llm_log10": c["llm_remainder_log10"],
                "linked_minus_control_llm": l["llm_remainder_log10"] - c["llm_remainder_log10"],
                "linked_zxcvbn_log10": l["zxcvbn_guesses_log10"],
                "control_zxcvbn_log10": c["zxcvbn_guesses_log10"],
            })
    df_pairs = pd.DataFrame(pair_rows)
    df_pairs.to_csv(RESULTS_DIR / "exp2_continuation_paired_summary.csv", index=False)

    if len(df_pairs):
        template_summary = (
            df_pairs.groupby(["template_id", "template_group"])
            .agg(
                n_pairs=("group_id", "count"),
                avg_linked_minus_control_llm=("linked_minus_control_llm", "mean"),
                median_linked_minus_control_llm=("linked_minus_control_llm", "median"),
                frac_linked_easier=("linked_minus_control_llm", lambda s: float((s < 0).mean())),
            )
            .reset_index()
        )
        # Larger score is better: control - linked.
        template_summary["avg_control_minus_linked_llm"] = -template_summary["avg_linked_minus_control_llm"]
        template_summary = template_summary.sort_values("avg_control_minus_linked_llm", ascending=False)
        template_summary.to_csv(RESULTS_DIR / "exp2_continuation_template_summary.csv", index=False)
        display(template_summary)

        plt.figure(figsize=(12, 5))
        plot_df = template_summary.sort_values("avg_control_minus_linked_llm", ascending=True)
        plt.barh(plot_df["template_id"], plot_df["avg_control_minus_linked_llm"])
        plt.xlabel("Average control - linked LLM log10 (higher = better separation)")
        plt.title("Continuation template separation")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "exp2_continuation_template_comparison.png", dpi=200)
        plt.show()
else:
    print("No continuation raw rows.")


In [ ]:
# !!!!!!!!! EVERYTHING BEYOND THIS POINT IS GARBAGE! DON'T ASK LLMS TO EVALUATE PASSWORD STRUCTUREDNESS DIRECTLY !!!!!!!!!.
# Structural yes/no scoring.
@torch.no_grad()
def completion_logprob(prompt, completion):
    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
    comp_ids = tokenizer(completion, add_special_tokens=False).input_ids
    if len(comp_ids) == 0:
        return -float("inf")
    ids = prompt_ids + comp_ids
    input_ids = torch.tensor([ids], device=model.device)
    out = model(input_ids=input_ids, use_cache=False)
    logits = out.logits[0]
    total = 0.0
    # token comp_ids[k] is predicted at position len(prompt_ids)+k-1
    for k, tok in enumerate(comp_ids):
        pos = len(prompt_ids) + k - 1
        if pos < 0:
            continue
        lp = torch.log_softmax(logits[pos].float(), dim=-1)[tok]
        total += float(lp.item())
    return total

def yes_no_probability(prompt):
    yes_lps = [completion_logprob(prompt, s) for s in YES_STRINGS]
    no_lps = [completion_logprob(prompt, s) for s in NO_STRINGS]
    ly = logsumexp(yes_lps)
    ln = logsumexp(no_lps)
    denom = logsumexp([ly, ln])
    return {
        "logp_yes_group": ly,
        "logp_no_group": ln,
        "p_yes_norm": math.exp(ly - denom) if math.isfinite(denom) else float("nan"),
        "p_no_norm": math.exp(ln - denom) if math.isfinite(denom) else float("nan"),
    }

structural_rows = []
if RUN_STRUCTURAL:
    total = len(df_pw) * len(structural_templates)
    pbar = tqdm(total=total, desc="Structural yes/no")
    t0 = time.perf_counter()
    for _, row in df_pw.iterrows():
        for tpl in structural_templates:
            prompt = safe_format_template(tpl["text"], password=row["password"])
            res = yes_no_probability(prompt)
            structural_rows.append({
                "model_alias": MODEL_ALIAS,
                "password_id": row["password_id"],
                "family": row["family"],
                "condition": row["condition"],
                "group_id": row["group_id"],
                "password": row["password"],
                "expected_structural_label": row["expected_structural_label"],
                "structure_type": row.get("structure_type", ""),
                "template_id": tpl["template_id"],
                "template_group": tpl["group"],
                "template_text": tpl["text"],
                **res,
            })
            pbar.update(1)
    pbar.close()
    print(f"Structural sweep finished in {time.perf_counter() - t0:.1f}s")

    df_struct_raw = pd.DataFrame(structural_rows)
    df_struct_raw.to_csv(RESULTS_DIR / "exp2_structural_raw.csv", index=False)
    display(df_struct_raw.head())
else:
    df_struct_raw = pd.DataFrame()
    print("RUN_STRUCTURAL=False")


In [ ]:
# Structural summaries.
if len(df_struct_raw):
    label_summary = (
        df_struct_raw.groupby(["template_id", "template_group", "expected_structural_label"])
        .agg(
            n=("password_id", "count"),
            mean_p_yes=("p_yes_norm", "mean"),
            median_p_yes=("p_yes_norm", "median"),
        )
        .reset_index()
    )
    label_summary.to_csv(RESULTS_DIR / "exp2_structural_template_label_summary.csv", index=False)
    display(label_summary)

    # Treat positive labels as structured, negative/random labels as non-structured. Ambiguous excluded from AUC.
    positive_labels = {"structured", "yes", "positive", "linked", "weak_pattern"}
    negative_labels = {"random", "negative", "no", "control_random", "probably_ok"}

    summary_rows = []
    for tpl_id, g in df_struct_raw.groupby("template_id"):
        template_group = g["template_group"].iloc[0]
        labels = g["expected_structural_label"].astype(str).str.lower()
        pos = g[labels.isin(positive_labels)]
        neg = g[labels.isin(negative_labels)]
        amb = g[~labels.isin(positive_labels | negative_labels)]
        row = {
            "template_id": tpl_id,
            "template_group": template_group,
            "n_pos": len(pos),
            "n_neg": len(neg),
            "n_ambiguous": len(amb),
            "mean_p_yes_pos": pos["p_yes_norm"].mean() if len(pos) else np.nan,
            "mean_p_yes_neg": neg["p_yes_norm"].mean() if len(neg) else np.nan,
            "mean_p_yes_ambiguous": amb["p_yes_norm"].mean() if len(amb) else np.nan,
            "false_positive_rate_at_0p8": float((neg["p_yes_norm"] >= 0.8).mean()) if len(neg) else np.nan,
            "false_negative_rate_at_0p8": float((pos["p_yes_norm"] < 0.8).mean()) if len(pos) else np.nan,
            "auc_pos_vs_neg": np.nan,
        }
        row["separation_pos_minus_neg"] = row["mean_p_yes_pos"] - row["mean_p_yes_neg"]
        if roc_auc_score is not None and len(pos) and len(neg):
            y = np.array([1] * len(pos) + [0] * len(neg))
            scores = np.concatenate([pos["p_yes_norm"].to_numpy(), neg["p_yes_norm"].to_numpy()])
            try:
                row["auc_pos_vs_neg"] = roc_auc_score(y, scores)
            except Exception:
                row["auc_pos_vs_neg"] = np.nan
        summary_rows.append(row)

    struct_summary = pd.DataFrame(summary_rows).sort_values(["auc_pos_vs_neg", "separation_pos_minus_neg"], ascending=False)
    struct_summary.to_csv(RESULTS_DIR / "exp2_structural_template_summary.csv", index=False)
    display(struct_summary)

    plt.figure(figsize=(12, 5))
    plot_df = struct_summary.sort_values("separation_pos_minus_neg", ascending=True)
    plt.barh(plot_df["template_id"], plot_df["separation_pos_minus_neg"])
    plt.xlabel("Mean P(yes) on structured - mean P(yes) on random/negative")
    plt.title("Structural template separation")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "exp2_structural_template_comparison.png", dpi=200)
    plt.show()
else:
    print("No structural raw rows.")


In [ ]:
# Recommended templates based on summaries.
recommendations = {}

cont_summary_path = RESULTS_DIR / "exp2_continuation_template_summary.csv"
struct_summary_path = RESULTS_DIR / "exp2_structural_template_summary.csv"

if cont_summary_path.exists():
    cs = pd.read_csv(cont_summary_path)
    if len(cs):
        best = cs.iloc[0].to_dict()
        recommendations["best_continuation_template"] = best
        print("Best continuation template:")
        display(pd.DataFrame([best]))

if struct_summary_path.exists():
    ss = pd.read_csv(struct_summary_path)
    if len(ss):
        best = ss.iloc[0].to_dict()
        recommendations["best_structural_template"] = best
        print("Best structural template:")
        display(pd.DataFrame([best]))

with open(RESULTS_DIR / "exp2_recommended_templates.json", "w") as f:
    json.dump(recommendations, f, indent=2)

print("Wrote", RESULTS_DIR / "exp2_recommended_templates.json")
print("All outputs in", RESULTS_DIR)
